# Análise de Desempenho Educacional - MA

> Objetivo: segmentar municípios por contexto socioeconômico e, dentro de cada grupo, identificar fatores associados aos destaques de IDEB.

Este notebook está alinhado com o script `src/analytics.py` e organizado em passos curtos.

Fluxo da análise:
1. Carregamento e checagem dos dados
2. Diagnóstico de clusterização (Elbow, Silhouette, PCA)
3. Clusterização final (k=2) e descrição dos grupos
4. Relevância de variáveis para diferenciar destaques no IDEB por cluster
5. Comparação de perfis e testes de Mann-Whitney
6. Gráficos finais para relatório

## 1) Carregamento e pré-processamento

> Neste passo você confere a base consolidada e prepara as variáveis para clustering.

Observação metodológica:
- Os clusters são formados apenas com variáveis socioeconômicas (`RENDA_PER_CAPITA_2022` e `PCT_POBREZA_EXTREMA_2022`).
- IDEB e demais variáveis entram depois para interpretação intracluster.

In [ ]:
import importlib
import pandas as pd
import analytics

importlib.reload(analytics)

print('Arquivo de entrada:', analytics.TRUSTED_DIR / 'indicadores_municipais_ma.csv')
df = analytics.carregar_dados()

print('Shape:', df.shape)
print('Municipios unicos:', df['NO_MUNICIPIO'].nunique())
print('\nVariaveis de cluster (socioeconomicas):', analytics.VARS_CLUSTER)
print('\nNulos nas variaveis de cluster:')
print(df[analytics.VARS_CLUSTER].isna().sum())

df[['NO_MUNICIPIO', 'RENDA_PER_CAPITA_2022', 'PCT_POBREZA_EXTREMA_2022', 'VL_OBSERVADO_2025']].head()

## 2) Diagnóstico de clusters

> Avalia diferentes valores de `k` e visualiza a separação dos grupos.

Leitura esperada:
- Elbow/Silhouette ajudam a justificar o número de clusters.
- PCA mostra se os grupos estão visualmente separáveis.

In [ ]:
X_scaled, _ = analytics.normalizar(df)
k_range = range(2, 11)

# Grafico Elbow + Silhouette (mostra em tela)
silhouettes = analytics.elbow_silhouette(X_scaled, k_range, salvar_arquivo=False)

# Visualizacao em PCA para o k configurado no pipeline
analytics.visualizar_pca(df, X_scaled, silhouettes, k_range, salvar_arquivo=False)

print('\nValor configurado no pipeline para cluster final:', analytics.CLUSTER_K)

## 3) Clusterização final e leitura dos grupos

> Executa a clusterização final (`k=2`) e lista os municípios destaque por IDEB em cada cluster.

Interpretação:
- `vulneráveis`: grupo com menor renda média
- `maior renda`: grupo com melhor contexto socioeconômico médio

In [ ]:
df_cluster = analytics.clusterizar(df.copy(), X_scaled, k=analytics.CLUSTER_K)

# Top e bottom IDEB dentro de cada cluster
analytics.destacar_municipios_por_cluster(df_cluster, top_n=5)

# Outliers positivos por cluster
analytics.outliers_positivos(df_cluster)

## 4) Variáveis mais relevantes para diferenciar destaques no IDEB

> Aqui a pergunta é: dentro de cada cluster, o que diferencia os municípios de IDEB alto vs baixo?

> Método padronizado no notebook:
- recorte por quantis (25% melhores vs 25% piores)
- Random Forest Classifier + filtro de significância (Mann-Whitney p < 0.10)

In [ ]:
# Relevancia das variaveis para a propria formacao dos clusters (socioeconomicas)
_ = analytics.relevancia_variaveis_cluster(X_scaled, k=analytics.CLUSTER_K, n_bootstrap=25)

# Relevancia de variaveis para diferenciar alto vs baixo IDEB dentro do cluster
analytics.variaveis_relevantes_por_cluster(
    df_cluster,
    salvar_arquivo=False,
    q=0.25,
    min_amostras_por_grupo=10,
 )

## 5) Comparação de perfis e testes estatísticos

> Este bloco usa o mesmo recorte por quantis para manter coerência com as sugestões de melhoria.

Saídas:
- médias por perfil (alto/médio/baixo) dentro de cada cluster
- p-valores do teste Mann-Whitney para cada variável explicativa

In [ ]:
analytics.comparar_perfis(df_cluster, q=0.25)
analytics.mann_whitney(df_cluster, q=0.25)

## 6) Gráficos finais (para relatório e apresentação)

> Este bloco gera e exibe os gráficos principais: distribuição do IDEB e importância global de variáveis.

Se quiser salvar todos os gráficos em `reports/figures`, execute a célula 7 com `salvar_graficos=True` no final.

In [ ]:
analytics.distribuicao_ideb(df_cluster, salvar_arquivo=False)
analytics.random_forest(df_cluster, salvar_arquivo=False)

# Opcional: rodar pipeline completo de uma vez e salvar figuras
# analytics.main(salvar_graficos=True)

In [ ]:
# Distribuição do IDEB no cluster vulnerável com cortes por quartis
# Gerado em: reports/figures/distribuicao_quartis_vulneraveis.png
analytics.distribuicao_quartis_cluster(df_cluster, cluster_nome='vulneráveis', q=0.25, salvar_arquivo=False)